[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/125/blob/main/07_dataframes/07_1_First_Look.ipynb)

# 07.1: A First Look at a DataFrame

In module 06 we kept related columns in separate series, and connected them by giving them the same index. A **DataFrame** is a single table that holds many columns together, and each column is a series. This notebook shows how to load a dataset into a DataFrame, how to check what it contains before you trust it, and how to work with its columns and rows.

This notebook opens in Google Colab. If Colab warns that the notebook was not authored by Google, click **Run anyway**. Your changes are not kept when you close the tab unless you choose **File > Save a copy in Drive**, so do that if you want to keep your work.

## Loading a file

`pd.read_csv()` reads a CSV file and returns a DataFrame. A CSV file is a plain text file where each line is a row and commas separate the columns. This one holds weekly sales for Broadway shows in New York. It is common to use the name `df` for a DataFrame.

In [ ]:
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_corgis_broadway.csv")
df.head()

`.head()` works just as it did with a series, and it shows the first five rows. Each row is **one show in one week**, so the first row is the play Tru at the Booth theatre, for the week of August 26, 1990. It took in a `Gross` of 134456 dollars. The bold numbers on the left are the index, and since we did not specify one, pandas numbered the rows from 0.

`.tail()` shows the last rows. It is worth checking both ends of a file, because stray notes and summary rows often collect at the bottom.

In [ ]:
df.tail(2)

The last rows are ordinary show-weeks for 2016, such as Wicked at the Gershwin, so nothing odd has collected at the end of this file.

## How big is it, and what is in it?

The first things to check on a new dataset are its size and its columns.

In [ ]:
df.shape

In [ ]:
df.columns

`.shape` gives 31296 rows and 12 columns, as a tuple with rows first. There are no parentheses after `.shape` and `.columns`, because they are attributes, like `.index` and `.values` on a series. `.info()` is a method that combines several of these checks.

In [ ]:
df.info()

For each column, `.info()` shows its name, how many values are not missing and its data type. Every column here has 31296 non-null values, so nothing is marked as missing. `int64` means whole numbers. Columns of text are listed with a text type, such as `object` or `str`, depending on your version of pandas. Check that each type is what you expected: if a column of numbers shows a text type, something in it is not a number.

`.dtypes` lists just the data types, which is useful when a table has many columns.

In [ ]:
df.dtypes.head(4)

`Attendance`, `Capacity` and `Day` are whole numbers, and `Full`, which holds dates such as 08/26/1990, is stored as text.

## Summarizing the numeric columns

`.describe()` on a DataFrame gives the summary from notebook 06.3 for each numeric column. Adding `.T` flips the table so that it is easier to read, with one row per column.

In [ ]:
df.describe().T

Look at `Gross`: the mean of about 596069 is above the median (the 50% column) of 514520, which is the sign of a long tail of very high weeks, and the max is 3201333. Now look at `Capacity`, which is the percentage of seats that were sold. Its median of 83 is believable, but its max of 892 cannot be right if it means a percentage of seats in a theatre. `Gross Potential` is also a percentage: it is the week's gross as a percentage of the most the show could have earned, given its ticket prices, seats and number of performances. This is why you read `.describe()` before you trust a column. Notebook 07.3 returns to problems like this one.

## Selecting columns

Put a column name in square brackets to get that column. The result is a series, so everything from module 06 works on it.

In [ ]:
print(type(df["Gross"]))
print(df["Gross"].max())
df["Type"].value_counts()

There are 22551 rows for musicals, 8406 for plays and 339 for specials. Keep in mind that these count show-weeks, not shows, because each row is one show in one week. A show that runs for years has many rows. For example, `.nunique()` tells us how many different shows there are.

In [ ]:
df["Name"].nunique()

There are 820 different shows among those 31296 rows. To select several columns, pass a **list** of names. This is why there are two sets of square brackets: the outer ones select, and the inner ones make the list. The result is a smaller DataFrame.

In [ ]:
df[["Name", "Type", "Gross"]].head(3)

## Adding a column

To add a column, assign to a new name. Math on columns works just as it does on a series, one row at a time. Dividing the gross by the attendance gives the average amount paid per ticket in each week.

In [ ]:
df["Ticket_Avg"] = df["Gross"] / df["Attendance"]
df[["Name", "Gross", "Attendance", "Ticket_Avg"]].head(3)

The first row is 134456 divided by 5500, or about 24.45 dollars per ticket. Unlike the math you saw in 06.2, this assignment changes `df` itself, since it now has a new column.

## Removing and renaming columns

`.drop(columns=[...])` returns a new DataFrame without those columns. `df` does not change unless you assign the result back to it. The column names themselves are an index of strings, so the `.str` methods from 06.4 work on them.

In [ ]:
smaller = df.drop(columns=["Day", "Month"])
print(smaller.shape)
print(df.shape)

In [ ]:
lower_case = df.copy()
lower_case.columns = lower_case.columns.str.lower()
lower_case.columns

`smaller` has 11 columns after dropping two, because `df` now has 13 with `Ticket_Avg`. To add a column with the same value in every row, assign one value, as in `df["Source"] = "CORGIS"`. To copy a column, assign the existing column to a new name, as in `df["Gross_Copy"] = df["Gross"]`. We made `lower_case` from a copy, so `df` keeps its original names.

`.rename()` changes particular column names and leaves the rest alone. You give it a dictionary of old names and new names.

In [ ]:
renamed = df.rename(columns={"Full": "Week", "Gross Potential": "Potential"})
renamed.columns[:6]

`Full` is now `Week` and `Gross Potential` is now `Potential`. Like `.drop()`, this returns a new DataFrame.

## Choosing the index

So far the index has been row numbers. `.set_index()` moves a column into the index, which lets you look rows up by that label, and `.reset_index()` moves it back to an ordinary column. Each returns a new DataFrame. Since many rows share a show name, `.loc` returns all of the matching rows, and they come back as a smaller table.

In [ ]:
by_show = df.set_index("Name")
print(by_show.index[:2])
print(by_show.loc["Tru", "Gross"])
print(by_show.loc["Wicked"].shape)
by_show.reset_index().columns[:2]

The first two labels are now Tru and Miss Saigon. Tru has only one row in this data, so its lookup gives the single gross of 134456. Wicked has 669 rows, so its lookup gives a smaller table of 669 rows and 12 columns. After `.reset_index()`, `Name` is back as an ordinary column, and it moves to the front of the table.

## Copies and aliases

The same rule applies here as with lists and series. `other = df` only makes a second name for the same table, while `.copy()` makes an independent one.

In [ ]:
other_name = df
independent = df.copy()
print(other_name is df)
print(independent is df)

The first line prints `True` and the second prints `False`, so a change made through `other_name` would also change `df`.

## Building a small table by hand

You will usually load a DataFrame from a file, but you can also build one yourself. `pd.DataFrame()` with no arguments makes an empty table. Given a dictionary, each key becomes a column name and each list becomes that column's values.

In [ ]:
scores = pd.DataFrame({"name": ["Ana", "Ben", "Cal"], "quiz": [9, 7, 10], "exam": [88, 75, 93]})
scores

This table has three rows and two numeric columns. It is also a handy size for showing how to work with rows. Assigning to a new label with `.loc` adds a row, and `.drop(index=...)` removes one.

In [ ]:
scores.loc[3] = ["Dee", 8, 81]
scores = scores.drop(index=1)
scores

Dee is now the row labeled 3, and Ben, who was the row labeled 1, is gone. Notice that the labels did not renumber themselves. Math works on several columns at once, and text methods work on a text column.

In [ ]:
scores[["quiz", "exam"]] = scores[["quiz", "exam"]] + 1
scores["name"] = scores["name"].str.upper()
scores

Every quiz and exam score went up by 1, and the names are now in upper case. Both changes are assigned back to the table, so `scores` itself changed.

## Practice problems

These problems are not collected or graded. They are here so that you can find out whether you can write this code yourself before the reading check and the pair activity. Write your answer in the cell under each problem, run it, and then run the check cell below it, which tells you whether your answer is right. The check cells are collapsed because they contain the answers, so try the problem before you open one.

Run the next two cells first. The first one creates fresh data for the problems, and the second sets up the checks.

In [ ]:
import pandas as pd

album_url = "https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_albums_sales_wikipedia.csv"
albums_start = pd.read_csv(album_url)

In [ ]:
#@title Run this cell to set up the checks
import numpy as np
import pandas as pd


def same(got, want):
    """True when got matches want: same labels and values for a series, same value for a number."""
    try:
        if isinstance(want, pd.DataFrame):
            if not isinstance(got, pd.DataFrame) or list(got.columns) != list(want.columns) or len(got) != len(want):
                return False
            if not pd.api.types.is_integer_dtype(want.index) and list(got.index) != list(want.index):
                return False
            got, want = got.reset_index(drop=True), want.reset_index(drop=True)
            for column in want.columns:
                if pd.api.types.is_numeric_dtype(want[column]):
                    if not np.allclose(pd.to_numeric(got[column], errors="coerce").astype(float), want[column].astype(float), equal_nan=True):
                        return False
                elif got[column].fillna("").astype(str).to_list() != want[column].fillna("").astype(str).to_list():
                    return False
            return True
        if isinstance(want, pd.Series):
            if not isinstance(got, pd.Series) or list(got.index) != list(want.index):
                return False
            if pd.api.types.is_numeric_dtype(want):
                return bool(np.allclose(got.astype(float), want.astype(float), equal_nan=True))
            return list(got) == list(want)
        if isinstance(want, (list, np.ndarray)):
            return list(got) == list(want)
        if isinstance(want, str):
            return got == want
        if isinstance(got, (pd.Series, pd.DataFrame, list, np.ndarray)):
            return False
        return bool(np.isclose(float(got), float(want)))
    except (TypeError, ValueError):
        return False


def check(number, name, want, hint, unchanged=None):
    """Print feedback for one problem. unchanged is (variable name, its starting value)."""
    if name not in globals():
        print(f"Problem {number}: not yet. No variable named {name} exists. Did you run your answer cell?")
    elif not same(globals()[name], want):
        print(f"Problem {number}: not yet. {hint}")
    elif unchanged is not None and not globals()[unchanged[0]].equals(unchanged[1]):
        print(f"Problem {number}: {name} is right, but {unchanged[0]} changed. Work on a copy, then run the setup cell again.")
    else:
        print(f"Problem {number}: correct.")

### Problem 1
Use `pd.read_csv()` with `album_url` to load a table of best-selling albums into a DataFrame named `albums`.

In [ ]:
# Problem 1

In [ ]:
#@title Check Problem 1
check(1, "albums", albums_start, "Pass album_url to pd.read_csv() and assign the result to albums.")

### Problem 2
Store the number of rows in `albums` in a variable named `n_rows`. Your code should get it from the table, not from counting.

In [ ]:
# Problem 2

In [ ]:
#@title Check Problem 2
check(2, "n_rows", len(albums_start), ".shape holds the rows and the columns, with rows first.")

### Problem 3
Create a series named `claims` that holds the `claim_sales` column.

In [ ]:
# Problem 3

In [ ]:
#@title Check Problem 3
check(3, "claims", albums_start["claim_sales"], "One column name in one set of square brackets gives a series.")

### Problem 4
Create a DataFrame named `short_table` that has only the columns `artist`, `album` and `year`, in that order.

In [ ]:
# Problem 4

In [ ]:
#@title Check Problem 4
check(4, "short_table", albums_start[["artist", "album", "year"]], "A list of column names takes two sets of square brackets.")

### Problem 5
Create a copy of `albums` named `with_gap` that has a new column, `gap`, equal to `claim_sales` minus `cert_sales`. `albums` should not change.

In [ ]:
# Problem 5

In [ ]:
#@title Check Problem 5
gap_want = albums_start.copy()
gap_want["gap"] = gap_want["claim_sales"] - gap_want["cert_sales"]
check(5, "with_gap", gap_want, "Use .copy(), then assign to the new column name.", ("albums", albums_start))

### Problem 6
Create a DataFrame named `no_genre` that is `albums` without the `genre` column.

In [ ]:
# Problem 6

In [ ]:
#@title Check Problem 6
check(6, "no_genre", albums_start.drop(columns=["genre"]), "Use .drop(columns=[...]).", ("albums", albums_start))

### Problem 7
Create a DataFrame named `renamed_albums` in which `cert_sales` is renamed to `certified`.

In [ ]:
# Problem 7

In [ ]:
#@title Check Problem 7
check(7, "renamed_albums", albums_start.rename(columns={"cert_sales": "certified"}), "Use .rename(columns={...}) with the old name and then the new name.")

### Problem 8
Create a DataFrame named `by_album` in which the `album` column has been moved into the index.

In [ ]:
# Problem 8

In [ ]:
#@title Check Problem 8
check(8, "by_album", albums_start.set_index("album"), "Use .set_index().")

## What's next

You can now load a dataset, check what is in it and work with its columns and rows. So far, though, most operations have used all 31296 rows. Most questions are about some of the rows, such as musicals in the year 2000. Notebook 07.2 shows how to select the rows you want.